# EDA parte 2 — Online Retail II
Continuación del análisis exploratorio. Ejecuta estas celdas **en el mismo Colab** donde ya tienes `df` cargado
(si reiniciaste el entorno, la primera celda lo vuelve a cargar desde `/content/online_retail_II.csv`).

Objetivos pendientes:
1. Separar devoluciones reales de ajustes de inventario
2. Identificar códigos que no son productos
3. Revisar valores atípicos (cantidad y precio)
4. Calcular ingresos y rankings
5. Decidir qué hacer con los duplicados
6. Construir el log de limpieza (cuántas filas se descartan en cada paso y por qué)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Si reiniciaste el entorno de Colab, vuelve a cargar los datos
if 'df' not in globals():
    df = pd.read_csv('/content/online_retail_II.csv')

df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['Invoice'] = df['Invoice'].astype(str)
df['StockCode'] = df['StockCode'].astype(str)

# Columnas auxiliares
df['es_cancelacion'] = df['Invoice'].str.startswith('C')
df['ingreso'] = df['Quantity'] * df['Price']

# Un código de producto "normal" son 5 dígitos, con letras opcionales al final (ej: 85123A)
PATRON_PRODUCTO = r'^\d{5}[A-Za-z]*$'
df['es_producto'] = df['StockCode'].str.match(PATRON_PRODUCTO)

print(df.shape)

## 1. Cantidades negativas: ¿devoluciones o ajustes de inventario?
Hipótesis: las facturas que empiezan con **C** son cancelaciones/devoluciones reales; las cantidades negativas
sin C probablemente son ajustes de inventario (precio 0).

In [ ]:
neg = df[df['Quantity'] < 0]
print("Filas con cantidad negativa:", len(neg))

print("\nSegún tipo de factura (True = empieza con C):")
print(neg['es_cancelacion'].value_counts())

sin_c = neg[~neg['es_cancelacion']]
print("\nNegativas SIN C, según precio:")
print((sin_c['Price'] == 0).value_counts().rename({True: 'Precio = 0', False: 'Precio > 0'}))

print("\nFacturas C con cantidad positiva:", (df['es_cancelacion'] & (df['Quantity'] > 0)).sum())

print("\nPrefijos de factura (NaN = factura solo numérica):")
print(df['Invoice'].str.extract(r'^([A-Za-z]+)')[0].value_counts(dropna=False))

In [ ]:
# Ejemplos de negativas sin C para revisar a ojo
sin_c.head(10)

## 2. Códigos de producto que no son productos
Códigos como POST (envío), M (manual), D (descuento), BANK CHARGES, etc. no son artículos vendidos.

In [ ]:
no_prod = df[~df['es_producto']]
print("Filas con código no-producto:", len(no_prod), f"({len(no_prod)/len(df):.2%})")

resumen = (no_prod.groupby('StockCode')
           .agg(filas=('StockCode', 'size'),
                descripcion=('Description', lambda s: s.dropna().iloc[0] if s.notna().any() else None),
                ingreso=('ingreso', 'sum'))
           .sort_values('filas', ascending=False))
resumen.head(25)

## 3. Descripciones sospechosas
En este dataset, las descripciones escritas a mano en minúscula suelen ser notas del personal ("damages", "lost", "mixed"), no productos.

In [ ]:
desc = df['Description'].dropna()
minusculas = df.loc[desc.index][desc.str.islower()]
print("Filas con descripción en minúscula:", len(minusculas))
print(minusculas['Description'].value_counts().head(20))

## 4. Definición de "venta válida" y valores atípicos
Venta válida = no cancelación, cantidad > 0, precio > 0, código de producto real.

In [ ]:
es_valida = (~df['es_cancelacion']) & (df['Quantity'] > 0) & (df['Price'] > 0) & df['es_producto']
ventas = df[es_valida].copy()
print("Ventas válidas:", len(ventas), f"({len(ventas)/len(df):.2%} del total)")

ventas[['Quantity', 'Price', 'ingreso']].describe(percentiles=[.5, .9, .99, .999])

In [ ]:
cols = ['Invoice', 'StockCode', 'Description', 'Quantity', 'Price', 'ingreso', 'Customer ID', 'InvoiceDate']
print("Top 10 por cantidad:")
display(ventas.nlargest(10, 'Quantity')[cols])
print("\nTop 10 por precio unitario:")
display(ventas.nlargest(10, 'Price')[cols])
print("\nTop 10 por ingreso de la línea:")
display(ventas.nlargest(10, 'ingreso')[cols])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(ventas['Quantity'].clip(upper=ventas['Quantity'].quantile(.99)), bins=50)
axes[0].set_title('Cantidad por línea (recortado al percentil 99)')
axes[1].hist(ventas['Price'].clip(upper=ventas['Price'].quantile(.99)), bins=50)
axes[1].set_title('Precio unitario (recortado al percentil 99)')
plt.tight_layout()
plt.show()

**Para decidir:** ¿hay líneas con cantidad enorme que tengan una cancelación casi inmediata (misma cantidad, factura C)?
Eso suele ser un error de digitación corregido después. Revisa abajo si los outliers de cantidad tienen su par.

In [ ]:
# Para cada outlier de cantidad, busca una cancelación del mismo producto con cantidad opuesta
top_q = ventas.nlargest(10, 'Quantity')
for _, fila in top_q.iterrows():
    par = df[(df['es_cancelacion']) &
             (df['StockCode'] == fila['StockCode']) &
             (df['Quantity'] == -fila['Quantity'])]
    print(f"{fila['StockCode']} qty={fila['Quantity']:>6} -> cancelaciones con cantidad opuesta: {len(par)}")

## 5. Ingresos y rankings
Sobre las ventas válidas.

In [ ]:
ingreso_total = ventas['ingreso'].sum()
print(f"Ingreso total (ventas válidas): {ingreso_total:,.2f}")

# Clientes sin ID
sin_id = ventas['Customer ID'].isna()
print(f"Líneas sin Customer ID: {sin_id.mean():.1%} | ingreso asociado: {ventas.loc[sin_id, 'ingreso'].sum() / ingreso_total:.1%}")

# UK vs internacional
es_uk = ventas['Country'] == 'United Kingdom'
print(f"Ingreso UK: {ventas.loc[es_uk, 'ingreso'].sum() / ingreso_total:.1%} | Internacional: {ventas.loc[~es_uk, 'ingreso'].sum() / ingreso_total:.1%}")

In [ ]:
# Ingreso mensual (el último mes es parcial: el dataset termina el 9-dic-2011)
mensual = ventas.groupby(ventas['InvoiceDate'].dt.to_period('M'))['ingreso'].sum()
colores = ['tab:blue'] * (len(mensual) - 1) + ['tab:orange']

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(mensual.index.astype(str), mensual.values, color=colores)
ax.set_title('Ingreso mensual (naranja = mes parcial)')
plt.xticks(rotation=60)
plt.tight_layout()
plt.show()

In [ ]:
# Top 10 productos por ingreso
top_prod = (ventas.groupby('StockCode')
            .agg(descripcion=('Description', lambda s: s.mode().iloc[0] if not s.mode().empty else None),
                 unidades=('Quantity', 'sum'),
                 ingreso=('ingreso', 'sum'))
            .sort_values('ingreso', ascending=False)
            .head(10))
display(top_prod)

# Top 10 clientes por ingreso (solo clientes identificados)
top_cli = (ventas.dropna(subset=['Customer ID'])
           .groupby('Customer ID')
           .agg(facturas=('Invoice', 'nunique'), ingreso=('ingreso', 'sum'))
           .sort_values('ingreso', ascending=False)
           .head(10))
display(top_cli)

## 6. Duplicados: ¿eliminar o conservar?
Un duplicado exacto = misma factura, producto, cantidad, precio, fecha y cliente. Puede ser un error del sistema
o dos unidades escaneadas por separado. Veamos cuánto pesan y dónde se concentran.

In [ ]:
dup = df.duplicated(keep='first')
print("Duplicados totales:", dup.sum())
print("Duplicados que son ventas válidas:", (dup & es_valida).sum())
print(f"Duplicados con Customer ID nulo: {df.loc[dup, 'Customer ID'].isna().mean():.1%}")

por_factura = dup.groupby(df['Invoice']).sum()
print("\nFacturas con al menos un duplicado:", (por_factura > 0).sum(), "de", df['Invoice'].nunique())
print("Duplicados por factura (solo facturas afectadas):")
print(por_factura[por_factura > 0].describe())

In [ ]:
# Ejemplos para inspeccionar visualmente
(df[df.duplicated(keep=False)]
 .sort_values(['Invoice', 'StockCode'])
 [['Invoice', 'StockCode', 'Description', 'Quantity', 'Price', 'InvoiceDate', 'Customer ID']]
 .head(12))

**Decisión sugerida:** eliminar los duplicados exactos de las ventas válidas y documentar el riesgo
(podrías estar subestimando ~3% de las líneas si algunas eran legítimas). Si estás de acuerdo, el log de limpieza ya lo aplica.

## 7. Log de limpieza (esto va al README)
Cuántas filas se descartan en cada paso y por qué. Las cancelaciones **no se tiran**: se guardan aparte como tabla de devoluciones.

In [ ]:
log = [{'paso': 'Datos originales', 'filas_removidas': 0, 'filas_restantes': len(df)}]
actual = df.copy()

def aplicar(nombre, mascara):
    global actual
    antes = len(actual)
    actual = actual[mascara(actual)]
    log.append({'paso': nombre, 'filas_removidas': antes - len(actual), 'filas_restantes': len(actual)})

aplicar('Quitar códigos que no son producto (POST, M, B, etc.)', lambda d: d['es_producto'])
aplicar('Quitar precio <= 0 (ajustes de inventario y deuda incobrable)', lambda d: d['Price'] > 0)

# Las cancelaciones se guardan aparte antes de sacarlas
devoluciones = actual[actual['es_cancelacion']].copy()
aplicar('Separar cancelaciones (facturas C) a tabla de devoluciones', lambda d: ~d['es_cancelacion'])

aplicar('Quitar cantidades <= 0 restantes', lambda d: d['Quantity'] > 0)
aplicar('Quitar duplicados exactos', lambda d: ~d.duplicated())

df_limpio = actual.copy()
df_limpio['cliente_identificado'] = df_limpio['Customer ID'].notna()

log_df = pd.DataFrame(log)
log_df['pct_del_original'] = (log_df['filas_removidas'] / len(df) * 100).round(2)
display(log_df)

print("\nVentas limpias:", len(df_limpio))
print("Devoluciones separadas:", len(devoluciones))

In [ ]:
# Verificación: los ingresos de las ventas limpias deben ser razonables y sin valores negativos
assert (df_limpio['Quantity'] > 0).all()
assert (df_limpio['Price'] > 0).all()
assert df_limpio['es_producto'].all()
print("Chequeos OK")
print(f"Ingreso total limpio: {df_limpio['ingreso'].sum():,.2f}")
print(f"Reembolsos (devoluciones): {devoluciones['ingreso'].sum():,.2f}")

In [ ]:
# Guardar resultados (descárgalos desde el panel de archivos de Colab o cópialos a Drive)
df_limpio.drop(columns=['es_cancelacion', 'es_producto']).to_csv('/content/retail_limpio.csv', index=False)
devoluciones.drop(columns=['es_producto']).to_csv('/content/retail_devoluciones.csv', index=False)
log_df.to_csv('/content/log_limpieza.csv', index=False)
print("Archivos guardados en /content/")

## Hallazgos — parte 2
*(Completa estos puntos con los números que te salieron al ejecutar las celdas; así quedan listos para el README.)*

- **Cantidades negativas:** ___ filas; de ellas ___ son cancelaciones (facturas C) y ___ son ajustes de inventario sin cliente.
- **Códigos no-producto:** ___ filas ( ___ %). Los más frecuentes: ___.
- **Outliers de cantidad/precio:** ___ (¿tienen cancelación par? ___).
- **Ingreso total (ventas válidas):** ___ . Sin Customer ID: ___ % del ingreso. UK: ___ % del ingreso.
- **Duplicados:** ___ eliminados; decisión: ___ .
- **Filas finales:** ___ de 1.067.371 ( ___ % del original).